# Liquidity & Momentum Terminal

Bloomberg-terminal-style diagnostic dashboard for FINRL asset features. The notebook uses the repository feature pipeline directly, so liquidity, accumulation, MACD, Klinger, RSI, Amihud, and volume diagnostics reflect the same trailing/no-lookahead calculations used elsewhere in the project.

This notebook is exploratory only. It does not modify PPO, DPO, environments, training code, or feature-engineering behavior.

In [76]:
from __future__ import annotations

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / "src"
for path in (PROJECT_ROOT, SRC_PATH):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    widgets = None
    HAS_WIDGETS = False
    from IPython.display import display

from finrl.data.download import download_ohlcv
from finrl.data.sources import MarketDataConfig
from finrl.data.universe import UniverseConfig
from finrl.features.asset import compute_asset_features
from finrl.features.columns import ACCUMULATION_FEATURE_COLUMNS, LIQUIDITY_EXIT_FEATURE_COLUMNS, selected_feature_indices
from finrl.features.preprocessing import PreprocessingConfig, fit_preprocessors, transform_features
from finrl.features.schema import FeatureConfig
from finrl.features.schema import FeatureBundle

pio.templates.default = "plotly_dark"
pd.set_option("display.max_columns", 120)

## Configuration

Edit these values, then run the notebook from top to bottom.

In [77]:
TICKERS = ("MU", "SPY", "QQQ")
START = "2024-01-01"
END = "2027-06-23"
CACHE_DIR = "data/cache"
LOCAL_OHLCV_PATH = None  # Example: "data/cache/ohlcv.parquet"
FEATURE_CONFIG = FeatureConfig(
    accumulation_window=60,
    liquidity_ratio_window=60,
)
PREPROCESSING_CONFIG = PreprocessingConfig(rolling_window=120)
USE_YFINANCE = True

In [78]:
EXPECTED_OHLCV_COLUMNS = {"date", "ticker", "open", "high", "low", "close", "adj_close", "volume"}

ACCUMULATION_EXPECTED = list(ACCUMULATION_FEATURE_COLUMNS)
LIQUIDITY_EXPECTED = list(LIQUIDITY_EXIT_FEATURE_COLUMNS)
MODEL_FEATURE_COLUMNS = [*ACCUMULATION_EXPECTED, *LIQUIDITY_EXPECTED]

def to_pandas_sorted(pl_df: pl.DataFrame) -> pd.DataFrame:
    df = pl_df.sort(["ticker", "date"]).to_pandas()
    df["date"] = pd.to_datetime(df["date"])
    return df

def make_asset_feature_bundle(features: pl.DataFrame) -> FeatureBundle:
    decision_dates = tuple(features.select(pl.col("date").cast(pl.Date)).unique().sort("date").to_series().to_list())
    tickers = tuple(features.get_column("ticker").unique().sort().to_list())
    asset_columns = tuple(column for column in features.columns if column not in {"date", "ticker"})
    empty_macro = pl.DataFrame({"date": decision_dates}).with_columns(pl.col("date").cast(pl.Date))
    empty_spectral = empty_macro.clone()
    return FeatureBundle(
        asset_features=features,
        macro_features=empty_macro,
        spectral_features=empty_spectral,
        decision_dates=decision_dates,
        tickers=tickers,
        asset_feature_columns=asset_columns,
        macro_feature_columns=(),
        spectral_feature_columns=(),
    )

def standardize_model_features(features: pl.DataFrame, config: PreprocessingConfig) -> FeatureBundle:
    bundle = make_asset_feature_bundle(features.select(["date", "ticker", *MODEL_FEATURE_COLUMNS]))
    fitted = fit_preprocessors(bundle, config)
    processed = transform_features(bundle, fitted)
    selected_feature_indices(processed.asset_feature_columns)
    return processed

def detect_feature_columns(features: pl.DataFrame) -> dict[str, list[str]]:
    columns = list(features.columns)
    expected = MODEL_FEATURE_COLUMNS
    detected = [col for col in expected if col in columns]
    return {
        "all": detected,
        "accumulation": [col for col in ACCUMULATION_EXPECTED if col in columns],
        "liquidity": [col for col in LIQUIDITY_EXPECTED if col in columns],
        "missing_expected": [col for col in expected if col not in columns],
    }

def zscore_by_ticker(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        values = out.groupby("ticker")[col]
        mean = values.transform("mean")
        std = values.transform("std").replace(0.0, np.nan)
        out[col] = (out[col] - mean) / std
    return out

def zscore_cross_section(df: pd.DataFrame, columns: list[str], date) -> pd.DataFrame:
    selected_date = pd.to_datetime(date)
    slice_df = df.loc[df["date"].eq(selected_date), ["ticker", *columns]].copy()
    for col in columns:
        std = slice_df[col].std()
        slice_df[col] = 0.0 if pd.isna(std) or std == 0 else (slice_df[col] - slice_df[col].mean()) / std
    return slice_df.replace([np.inf, -np.inf], np.nan).fillna(0.0)

def apply_rolling(df: pd.DataFrame, columns: list[str], window: int) -> pd.DataFrame:
    if not window or window <= 1:
        return df.copy()
    out = df.copy()
    out[columns] = out.groupby("ticker", group_keys=False)[columns].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    return out

def available(columns: list[str], df: pd.DataFrame) -> list[str]:
    return [col for col in columns if col in df.columns]

def ticker_frame(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    return df.loc[df["ticker"].eq(ticker)].sort_values("date")

def plot_price_volume(df: pd.DataFrame, ticker: str) -> go.Figure:
    one = ticker_frame(df, ticker)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.04, row_heights=[0.7, 0.3], specs=[[{}], [{"secondary_y": True}]])
    fig.add_trace(go.Candlestick(x=one["date"], open=one["open"], high=one["high"], low=one["low"], close=one["close"], name="OHLC"), row=1, col=1)
    fig.add_trace(go.Bar(x=one["date"], y=one["volume"], name="Volume", marker_color="#5b8ff9", opacity=0.55), row=2, col=1)
    if "dollar_volume" in one.columns:
        fig.add_trace(go.Scatter(x=one["date"], y=one["dollar_volume"], name="Dollar volume", line=dict(color="#f6bd16", width=1.4)), row=2, col=1, secondary_y=True)
    fig.update_layout(title=f"{ticker} Price / Volume", height=720, xaxis_rangeslider_visible=False, legend_orientation="h")
    return fig

def plot_feature_lines(df: pd.DataFrame, ticker: str, features: list[str], normalize: bool = False, rolling_window: int | None = None) -> go.Figure:
    cols = available(features, df)
    one = ticker_frame(df, ticker)
    if rolling_window and rolling_window > 1:
        one = apply_rolling(one, cols, rolling_window)
    if normalize and cols:
        one = zscore_by_ticker(one, cols)
    fig = go.Figure()
    for col in cols:
        fig.add_trace(go.Scatter(x=one["date"], y=one[col], mode="lines", name=col, hovertemplate="%{x|%Y-%m-%d}<br>%{y:.4f}<extra>" + col + "</extra>"))
    fig.update_layout(title=f"{ticker} Feature Lines", height=520, hovermode="x unified", legend_orientation="h")
    return fig

def plot_grouped_terminal(df: pd.DataFrame, ticker: str, groups: list[tuple[str, list[str]]], normalize: bool = False, rolling_window: int | None = None, log_y: bool = False) -> go.Figure:
    row_count = len(groups)
    fig = make_subplots(rows=row_count, cols=1, shared_xaxes=True, vertical_spacing=0.04, subplot_titles=[name for name, _ in groups])
    for idx, (_, cols) in enumerate(groups, start=1):
        subset = ticker_frame(df, ticker)
        present = available(cols, subset)
        if rolling_window and rolling_window > 1:
            subset = apply_rolling(subset, present, rolling_window)
        if normalize and present:
            subset = zscore_by_ticker(subset, present)
        for col in present:
            fig.add_trace(go.Scatter(x=subset["date"], y=subset[col], mode="lines", name=col), row=idx, col=1)
        if log_y:
            fig.update_yaxes(type="log", row=idx, col=1)
    fig.update_layout(height=max(360, 260 * row_count), hovermode="x unified", legend_orientation="h")
    return fig

def latest_snapshot(df: pd.DataFrame) -> pd.DataFrame:
    cols = available(["close", "return", "volume", "dollar_volume", "acc_rsi", "acc_price_drift", "liq_amihud_illiquidity", "liquidity_ratio", "liquidity_deterioration"], df)
    latest_idx = df.sort_values("date").groupby("ticker").tail(1).index
    out = df.loc[latest_idx, ["date", "ticker", *cols]].sort_values("ticker").reset_index(drop=True)
    return out

def plot_cross_section_heatmap(df: pd.DataFrame, columns: list[str], date) -> go.Figure:
    cols = available(columns, df)
    zdf = zscore_cross_section(df, cols, date)
    matrix = zdf.set_index("ticker")[cols]
    fig = px.imshow(matrix, aspect="auto", color_continuous_scale="RdBu", color_continuous_midpoint=0.0, labels=dict(color="z-score"))
    fig.update_layout(title=f"Cross-Sectional Feature Heatmap: {pd.to_datetime(date).date()}", height=max(420, 24 * len(cols)))
    return fig

def plot_feature_ranking(df: pd.DataFrame, feature: str, ascending: bool = False) -> go.Figure:
    latest = df.sort_values("date").groupby("ticker").tail(1)
    ranked = latest[["ticker", feature]].dropna().sort_values(feature, ascending=ascending)
    fig = px.bar(ranked, x=feature, y="ticker", orientation="h", title=f"Latest Ranking: {feature}", color=feature, color_continuous_scale="Tealrose")
    fig.update_layout(height=max(360, 42 * max(1, len(ranked))))
    return fig

## Data Loading

The preferred path uses FINRL market data APIs. If yfinance is unavailable, the notebook falls back to `LOCAL_OHLCV_PATH` or `data/cache/ohlcv.parquet`.

In [79]:
universe = UniverseConfig(tickers=TICKERS, include_cash=False, benchmark_ticker="SPY")
source_config = MarketDataConfig(
    universe=universe,
    start=START,
    end=END,
    cache_dir=CACHE_DIR,
    source="yfinance",
    auto_adjust=False,
    actions=False,
    macro_tickers=(),
)

fallback_paths = [Path(LOCAL_OHLCV_PATH)] if LOCAL_OHLCV_PATH else []
fallback_paths.append(Path(CACHE_DIR) / "ohlcv.parquet")

load_error: Exception | None = None
ohlcv: pl.DataFrame | None = None
if USE_YFINANCE:
    try:
        ohlcv = download_ohlcv(TICKERS, START, END, source_config)
    except Exception as exc:
        load_error = exc
        print(f"yfinance download failed: {exc}")

if ohlcv is None:
    for path in fallback_paths:
        if path.exists():
            print(f"Loading local OHLCV parquet: {path}")
            ohlcv = pl.read_parquet(path)
            break

if ohlcv is None:
    raise RuntimeError("No OHLCV data loaded. Enable yfinance or set LOCAL_OHLCV_PATH to a parquet file.") from load_error

missing_ohlcv = sorted(EXPECTED_OHLCV_COLUMNS.difference(ohlcv.columns))
if missing_ohlcv:
    raise ValueError(f"OHLCV frame is missing required columns: {missing_ohlcv}")

ohlcv = ohlcv.select(["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]).sort(["ticker", "date"])
ohlcv_pd = to_pandas_sorted(ohlcv)
print(ohlcv.shape)
ohlcv.head()

(1860, 8)


date,ticker,open,high,low,close,adj_close,volume
date,str,f64,f64,f64,f64,f64,i64
2024-01-02,"""MU""",84.0,84.080002,81.75,82.339996,81.687599,13597100
2024-01-03,"""MU""",81.199997,82.589996,80.580002,82.260002,81.608246,12915600
2024-01-04,"""MU""",83.470001,84.260002,82.610001,82.709999,82.05468,19134200
2024-01-05,"""MU""",81.480003,83.529999,81.010002,83.449997,82.788818,15483700
2024-01-08,"""MU""",83.889999,85.510002,83.830002,84.949997,84.276917,16219800


## Feature Generation & Discovery

Feature values are produced by `compute_asset_features`; visualization code only selects, reshapes, normalizes, and plots them.

In [80]:
raw_features = compute_asset_features(ohlcv, FEATURE_CONFIG)
feature_groups = detect_feature_columns(raw_features)
standardized_bundle = standardize_model_features(raw_features, PREPROCESSING_CONFIG)
features = standardized_bundle.asset_features
features_pd = to_pandas_sorted(features)

print(f"Total standardized model feature count: {len(feature_groups['all'])}")
print(f"Rolling standardization window: {PREPROCESSING_CONFIG.rolling_window} rows per ticker")
print("\nStandardized model-facing feature columns:")
for col in feature_groups["all"]:
    print(f"- {col}")

print("\nMissing model-facing columns:")
if feature_groups["missing_expected"]:
    for col in feature_groups["missing_expected"]:
        print(f"- {col}")
else:
    print("None")

Total standardized model feature count: 17
Rolling standardization window: 120 rows per ticker

Standardized model-facing feature columns:
- acc_price_drift
- acc_liquidity_growth
- acc_vol_compression
- acc_low_vol
- acc_macd_improvement
- acc_klinger_improvement
- acc_macd_bullish_hist
- acc_klinger_bullish_hist
- acc_macd_early
- acc_klinger_early
- acc_momentum_quality
- liq_amihud_trend
- liq_liquidity_deterioration
- liq_klinger_deterioration
- liq_vol_expansion
- liq_liquidity_shock
- liq_momentum_quality

Missing model-facing columns:
None


## Latest Market Snapshot

In [81]:
snapshot = features_pd.sort_values("date").groupby("ticker").tail(1)[["date", "ticker", *MODEL_FEATURE_COLUMNS]].sort_values("ticker").reset_index(drop=True)
try:
    display(snapshot.style.format(precision=4).background_gradient(cmap="viridis", subset=snapshot.select_dtypes(include=np.number).columns))
except Exception:
    display(snapshot)

,date,ticker,acc_price_drift,acc_liquidity_growth,acc_vol_compression,acc_low_vol,acc_macd_improvement,acc_klinger_improvement,acc_macd_bullish_hist,acc_klinger_bullish_hist,acc_macd_early,acc_klinger_early,acc_momentum_quality,liq_amihud_trend,liq_liquidity_deterioration,liq_klinger_deterioration,liq_vol_expansion,liq_liquidity_shock,liq_momentum_quality
0,2026-06-23,MU,1.406420,1.012591,-2.315199,-2.830789,0.173932,0.305784,0.097369,-0.506073,-0.742768,1.308921,-0.479505,1.956686,-1.012591,-0.305784,2.315199,-0.069878,-0.479505
1,2026-06-23,QQQ,1.626502,-0.318044,-1.319269,-3.511596,-0.108245,-0.050352,-1.118364,-1.157789,-0.652810,0.592785,-0.476644,0.897652,0.318044,0.050352,1.319269,-0.624802,-0.476644
2,2026-06-23,SPY,2.075288,-0.311902,-0.147732,-1.311741,0.019434,-0.444630,-1.105560,-1.235329,-0.009327,2.772956,-0.657737,0.054521,0.311902,0.444630,0.147732,-0.830887,-0.657737


## Price and Volume Overview

In [82]:
DEFAULT_TICKER = TICKERS[0]
plot_price_volume(ohlcv_pd, DEFAULT_TICKER).show()
plot_feature_lines(features_pd, DEFAULT_TICKER, MODEL_FEATURE_COLUMNS, normalize=False, rolling_window=None).show()


## Momentum Terminal

In [83]:
MOMENTUM_FEATURES = list(ACCUMULATION_FEATURE_COLUMNS)
plot_feature_lines(features_pd, DEFAULT_TICKER, MOMENTUM_FEATURES, normalize=False, rolling_window=None).show()

## MACD and Klinger Terminal

In [84]:
MACD_KLINGER_GROUPS = [
    ("Model MACD Inputs", ["acc_macd_improvement", "acc_macd_bullish_hist", "acc_macd_early"]),
    ("Model Klinger Inputs", ["acc_klinger_improvement", "acc_klinger_bullish_hist", "acc_klinger_early"]),
]
plot_grouped_terminal(features_pd, DEFAULT_TICKER, MACD_KLINGER_GROUPS).show()

## Liquidity Terminal

In [85]:
LIQUIDITY_GROUPS = [
    ("Model Liquidity-Exit Inputs", list(LIQUIDITY_EXIT_FEATURE_COLUMNS)),
]
plot_grouped_terminal(features_pd, DEFAULT_TICKER, LIQUIDITY_GROUPS).show()

## Amihud Terminal

In [86]:
AMIHUD_GROUPS = [
    ("Model Amihud Input", ["liq_amihud_trend"]),
]
plot_grouped_terminal(features_pd, DEFAULT_TICKER, AMIHUD_GROUPS, log_y=False).show()

## Cross-Sectional Heatmap

In [87]:
LATEST_DATE = features_pd["date"].max()
plot_cross_section_heatmap(features_pd, feature_groups["all"], LATEST_DATE).show()

## Feature Ranking Panel

In [88]:
DEFAULT_FEATURE = "acc_price_drift" if "acc_price_drift" in features_pd.columns else feature_groups["all"][0]
plot_feature_ranking(features_pd, DEFAULT_FEATURE, ascending=False).show()

## Single-Ticker Multi-Feature Comparison

In [89]:
DEFAULT_MULTI_FEATURES = available(MODEL_FEATURE_COLUMNS[:8], features_pd)
plot_feature_lines(features_pd, DEFAULT_TICKER, DEFAULT_MULTI_FEATURES, normalize=False, rolling_window=5).show()

## Interactive Dashboard

If `ipywidgets` is installed, this section exposes ticker, feature, date, smoothing, normalization, and log-scale controls. Otherwise the static charts above remain available.

In [90]:
if not HAS_WIDGETS:
    print("ipywidgets is not installed. Static charts were generated above; install ipywidgets to enable interactive controls.")
else:
    ticker_widget = widgets.Dropdown(options=sorted(features_pd["ticker"].unique()), value=DEFAULT_TICKER, description="Ticker")
    feature_widget = widgets.Dropdown(options=feature_groups["all"], value=DEFAULT_FEATURE, description="Rank")
    multi_widget = widgets.SelectMultiple(options=feature_groups["all"], value=tuple(DEFAULT_MULTI_FEATURES[:4]), description="Features", rows=10)
    date_widget = widgets.SelectionSlider(options=sorted(features_pd["date"].dt.date.unique()), value=LATEST_DATE.date(), description="Date", continuous_update=False)
    smoothing_widget = widgets.IntSlider(value=1, min=1, max=60, step=1, description="Smooth")
    rolling_widget = widgets.Checkbox(value=True, description="Rolling")
    log_widget = widgets.Checkbox(value=False, description="Log y")
    ascending_widget = widgets.Checkbox(value=False, description="Ascending")

    def render_terminal(ticker: str, rank_feature: str, selected_features: tuple[str, ...], selected_date, smooth: int, rolling: bool, log_y: bool, ascending: bool):
        window = smooth if rolling else None
        display(plot_price_volume(ohlcv_pd, ticker))
        display(plot_feature_lines(features_pd, ticker, list(selected_features), normalize=False, rolling_window=window))

    controls = widgets.VBox([
        widgets.HBox([ticker_widget, feature_widget, date_widget]),
        widgets.HBox([smoothing_widget, rolling_widget, log_widget, ascending_widget]),
        multi_widget,
    ])
    out = widgets.interactive_output(render_terminal, {
        "ticker": ticker_widget,
        "rank_feature": feature_widget,
        "selected_features": multi_widget,
        "selected_date": date_widget,
        "smooth": smoothing_widget,
        "rolling": rolling_widget,
        "log_y": log_widget,
        "ascending": ascending_widget,
    })
    display(controls, out)


Output()